# Kapitel 5 Begleit-Notebook
**Bauen Sie Ihr erstes LLM — Kapitel 5: Ihr erstes Python-Programm**

Dieses Notebook bündelt die ausführbaren Codebeispiele aus Kapitel 5. Führen Sie die Zellen von oben nach unten aus.

- Installationen: transformers (für GPT-2 Demo)
- Daten: kleine Inline-Strings; keine externen Dateien erforderlich
- Laufzeit: CPU ist ausreichend; GPU beschleunigt nur den GPT-2-Aufruf

In [ ]:
# ===== SETUP =====
# Installiere transformers Bibliothek (HuggingFaces Toolkit für die Arbeit mit LLMs)
!pip install -q transformers==4.46.1

import warnings
warnings.filterwarnings('ignore')  # Unterdrücke kleinere Versionswarnungen

# Lade HuggingFace-Tools für GPT-2
# "pipeline" ist ein Helfer, der ein Modell lädt und alle Komplexität für uns verarbeitet
# Stellen Sie es sich als vorgefertigten Workflow vor: Modell laden → Eingabe verarbeiten → Ausgabe generieren
from transformers import pipeline, logging
logging.set_verbosity_error()  # Zeige nur echte Fehler, keine Info-Meldungen

print('Setup abgeschlossen')

## Schneller Erfolg: GPT-2 Textgenerierung
Führen Sie eine kleine GPT-2-Generierung aus, um ein LLM in Aktion zu sehen.

In [ ]:
# Lade GPT-2 Textgenerierungsmodell (124M Parameter)
generator = pipeline('text-generation', model='gpt2')

# Generiere Text aus einem Startprompt
result = generator(
    'The secret to building AI is',  # Starttext
    max_new_tokens=20,                # Generiere 20 weitere Wörter
    do_sample=True,                   # Verwende Zufälligkeit (nicht nur wahrscheinlichste Wörter)
    pad_token_id=50256                # Technisch: vermeidet eine Warnung
)

# Extrahiere und gebe den generierten Text aus
print(result[0]['generated_text'])

## Strings und grundlegende Operationen
Arbeiten mit Text, Längen und Slices.

**Python-Tipp: f-strings** ermöglichen es Ihnen, Variablenwerte in Text einzufügen. Das `f` vor dem String steht für "formatiert", und geschweifte Klammern `{}` markieren, wo Werte eingefügt werden sollen:
```python
name = "GPT"
print(f"Hallo, {name}!")  # → Hallo, GPT!
```

In [ ]:
# Beispielausgabe von einem Textgenerierungsmodell
output = 'The secret to building AI is understanding how machines learn from data'
prompt = 'The secret to building AI is'

# Grundlegende String-Operationen
print(len(output))  # Länge in Zeichen
print(type(output))  # Bestätige, dass es ein String ist

# Teile in Wörter auf (Liste von Strings)
words = output.split()
print(words)

# Methodenverkettung: rufe mehrere Operationen nacheinander auf
# Dies ist äquivalent zu zwei Zeilen:
#   lowercase_text = output.lower()
#   words = lowercase_text.split()
words = output.lower().split()
print(words)

# Extrahiere nur den generierten Teil (Slice ab Ende des Prompts)
generated = output[len(prompt):]
print(f'Generiert: {generated.strip()}')  # .strip() entfernt führende/nachfolgende Leerzeichen
print(f'Wortanzahl: {len(generated.split())}')

## Zahlen und Formatierung
Grundlegende numerische Werte und f-strings.

In [ ]:
# Python verwendet Unterstriche für bessere Lesbarkeit bei großen Zahlen
num_parameters = 124_000_000  # 124 Millionen (GPT-2-Größe)
learning_rate = 0.0001        # Kleine Schrittgröße für Training
vocab_size = 50257            # GPT-2s Vokabulargröße

# f-string Formatierungstricks:
print(f'GPT-2 hat {num_parameters:,} Parameter')   # :, fügt Tausendertrennzeichen hinzu
print(f'Lernrate ist {learning_rate:.2e}')         # :.2e = wissenschaftliche Notation
print(f'Halbes Vokabular: {vocab_size // 2}')      # Sie können innerhalb von {} rechnen!

## Baue ein kleines Vokabular und einen Tokenizer
Von Spielzeugsätzen zu einem Tokenizer auf Wortebene.

In [ ]:
# Beispiel-Textdaten (worauf Sie ein Modell trainieren würden)
texts = [
    'The secret to building AI is',
    'The key to machine learning is data',
    'AI systems learn from examples'
]

# Sammle alle Wörter aus allen Texten
all_words = []
for text in texts:
    words = text.lower().split()  # Normalisiere zu Kleinbuchstaben
    # extend() fügt jedes Element einzeln zur Liste hinzu
    # (append() würde die gesamte Liste als EIN Element hinzufügen)
    all_words.extend(words)

print(all_words)

# Listen-Slicing-Beispiele
print(all_words[0], all_words[-1], all_words[:3])  # Erstes, letztes, erste 3

# Baue Vokabular: ordne jedem eindeutigen Wort eine Nummer zu
vocab = {'<PAD>': 0, '<UNK>': 1}  # Spezielle Tokens zuerst (reservierte IDs)

for word in all_words:
    if word not in vocab:
        # len(vocab) gibt die "nächste verfügbare ID"
        # Wenn vocab 2 Elemente hat (IDs 0 und 1), wird len(vocab)=2 zur nächsten ID
        vocab[word] = len(vocab)

print(f'Vokabulargröße: {len(vocab)}')
print(vocab)

# Schlage Wörter im Vokabular nach
print(vocab['the'], vocab['ai'])  # Gibt ihre IDs zurück

## Vergleich mit GPT-2 Tokenizer
Zeigen Sie, wie sich ein Produktions-Tokenizer unterscheidet.

**Python-Tipps:**
- `dict.get(key, default)` gibt den Wert für `key` zurück, falls vorhanden, andernfalls `default`. Sicherer als `dict[key]`, das abstürzt, wenn der Schlüssel fehlt.
- **List Comprehension** ist eine kompakte Möglichkeit, Listen zu erstellen. `[expr for item in collection]` ist äquivalent zu einer For-Schleife, die an eine Liste anhängt.

In [ ]:
# Lade den echten GPT-2 Tokenizer
from transformers import GPT2Tokenizer
real_tok = GPT2Tokenizer.from_pretrained('gpt2')

# Vergleiche Vokabulargrößen
print(f'Unser Vokabular: {len(vocab)} Wörter')
print(f'GPT-2 Vokabular: {len(real_tok)} Tokens')

# Teste Handhabung unbekannter Wörter
# .get(key, default) gibt default zurück, wenn Schlüssel nicht gefunden (statt abzustürzen)
word = 'neural'
print(f"'{word}' → {vocab.get(word, vocab['<UNK>'])}")  # Gibt <UNK> ID (1) zurück

# Tokenisiere einen Satz mit unserem Vokabular
sentence = 'The neural network learns'

# Lange Version (explizite Schleife):
token_ids = []
for word in sentence.lower().split():
    token_id = vocab.get(word, vocab['<UNK>'])  # Hole ID oder <UNK> falls unbekannt
    token_ids.append(token_id)
    print(f'  {word} → {token_id}')

print(f'Token-IDs (Schleife): {token_ids}')

# Kurze Version (List Comprehension - gleiches Ergebnis, kompakter):
token_ids = [vocab.get(w, vocab['<UNK>']) for w in sentence.lower().split()]
print(f'Token-IDs (Comprehension): {token_ids}')

## Tokenisierungs- und Detokenisierungs-Helfer
Rundreise eines Satzes.

In [ ]:
# Funktion: Text → Token-IDs (Kodierung)
def tokenize(text, vocab):
    words = text.lower().split()
    return [vocab.get(w, vocab['<UNK>']) for w in words]

# Funktion: Token-IDs → Text (Dekodierung)
def detokenize(ids, vocab):
    # Erstelle Umkehr-Mapping (ID → Wort)
    # Dictionary Comprehension: {new_key: new_val for key, val in dict.items()}
    # vocab.items() gibt Paare wie ('the', 2), ('secret', 3), etc. zurück
    # Wir drehen sie um: (2, 'the'), (3, 'secret'), etc.
    id_to_word = {v: k for k, v in vocab.items()}
    return ' '.join(id_to_word.get(i, '<UNK>') for i in ids)

# Teste Rundreise: Text → IDs → Text
ids = tokenize('The secret to AI', vocab)
print(f'Kodiert: {ids}')
print(f'Dekodiert: {detokenize(ids, vocab)}')

# Vergleiche unseren Tokenizer mit GPT-2s
text = 'The secret to AI'
print(f'Unsere Tokens:   {tokenize(text, vocab)}')
print(f'GPT-2 Tokens: {real_tok.encode(text)}')  # Unterschiedlich! GPT-2 verwendet Subwords, keine ganzen Wörter

## Eine minimale Tokenizer-Klasse
Zustandsbehafteter Tokenizer auf Wortebene mit fit/encode/decode.

**Python-Klassen 101:**
Eine **Klasse** ist ein Bauplan zur Erstellung von Objekten, die Daten und Funktionen zusammenfassen.

- `class MyClass:` — definiert den Bauplan
- `__init__(self)` — wird ausgeführt, wenn Sie ein neues Objekt erstellen (initialisiert seine Daten)
- `self` — bezieht sich auf "dieses spezifische Objekt" (wie "dieses Auto" vs. "Autos im Allgemeinen")
- Methoden (Funktionen innerhalb einer Klasse) erhalten automatisch `self` als ersten Parameter

**Analogie:** Eine Klasse ist wie ein Auto-Bauplan. `__init__` richtet anfängliche Merkmale ein (Farbe, Motorgröße). Wenn Sie ein Auto aus dem Bauplan bauen, bezieht sich `self` auf DIESES spezifische Auto.

In [ ]:
# Objektorientierter Tokenizer (Klasse bündelt Daten + Methoden)
class SimpleTokenizer:
    def __init__(self):
        # Initialisiere Vokabular mit speziellen Tokens
        self.word_to_id = {'<PAD>': 0, '<UNK>': 1}
        self.id_to_word = {0: '<PAD>', 1: '<UNK>'}

    def fit(self, texts):
        """Baue Vokabular aus Trainingstexten"""
        for text in texts:
            for word in text.lower().split():
                if word not in self.word_to_id:
                    idx = len(self.word_to_id)
                    self.word_to_id[word] = idx  # Füge neues Wort hinzu
                    self.id_to_word[idx] = word  # Umkehr-Mapping

    def encode(self, text):
        """Konvertiere Text zu Token-IDs"""
        return [self.word_to_id.get(w, 1) for w in text.lower().split()]  # 1 = <UNK>

    def decode(self, ids):
        """Konvertiere Token-IDs zurück zu Text"""
        return ' '.join(self.id_to_word.get(i, '<UNK>') for i in ids)

    def __len__(self):
        """Gebe Vokabulargröße zurück (ermöglicht len(tok))"""
        return len(self.word_to_id)

# Erstelle und trainiere Tokenizer
tok = SimpleTokenizer()
tok.fit(texts)  # Lerne Vokabular aus unseren Trainingsdaten

print(f'Vokabulargröße: {len(tok)}')

# Teste Kodierung/Dekodierung
ids = tok.encode('The secret to AI')
print(f'Kodiert: {ids}')
print(f'Dekodiert: {tok.decode(ids)}')

# Vergleiche noch einmal mit GPT-2
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')
text = 'The secret to AI'
print(f'Ihr Tokenizer:   {tok.encode(text)}')
print(f'GPT-2 Tokenizer: {gpt2_tok.encode(text)}')  # GPT-2 verwendet Byte-Pair Encoding (BPE)

## Voller Kreis: Ihr Tokenizer vs GPT-2

Vergleichen wir Ihren handgebauten Tokenizer ein letztes Mal mit dem echten GPT-2-Tokenizer. Beachten Sie, wie GPT-2s Token-IDs viel größere Zahlen sind (es hat 50.257 Tokens!) und es **Subword-Tokenisierung** (BPE) statt ganzer Wörter verwendet.

In [ ]:
# Erstelle einen frischen Tokenizer und vergleiche mit GPT-2
my_tok = SimpleTokenizer()
my_tok.fit(['The secret to building AI is understanding'])

# Lade GPT-2s Tokenizer
from transformers import GPT2Tokenizer
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')

# Vergleiche am gleichen Text
text = 'The secret to AI'
print(f'Ihr Tokenizer:   {my_tok.encode(text)}')
print(f'GPT-2 Tokenizer: {gpt2_tok.encode(text)}')

print(f'\nIhre Vokabulargröße:  {len(my_tok)}')
print(f'GPT-2 Vokabulargröße: {len(gpt2_tok)}')

## Was ist gerade passiert?

Sie haben einen vollständigen Tokenizer von Grund auf gebaut! Hier ist, was Sie gelernt haben:

1. **Strings** — Textmanipulation mit `.split()`, `.lower()`, Slicing
2. **Dictionaries** — Schlüssel-Wert-Zuordnungen für Vokabular-Lookup
3. **Listen** — Geordnete Sammlungen für Token-Sequenzen  
4. **Funktionen** — Wiederverwendbare Codeblöcke (`def tokenize(...)`)
5. **Klassen** — Baupläne, die Daten + Methoden zusammenfassen

**Wichtige Erkenntnis:** Ihr Tokenizer verwendet ganze Wörter, daher werden unbekannte Wörter zu `<UNK>`. GPT-2 verwendet **Byte-Pair Encoding (BPE)**, das Wörter in Subwords zerlegt — deshalb sieht es selten wirklich unbekannte Tokens. Sie werden mehr darüber in Kapitel 8 lernen!